# 02 — EDA: Telemetria e Análise de Don't Go

**Objetivo:** Explorar os 37M+ eventos de alarme, entender a distribuição de criticidade, identificar os alarmes-chave associados ao Don't Go e validar as hipóteses H1–H7.

**Dataset:** 6 arquivos parquet mensais | Jan–Jun 2025 | 35 equipamentos | Mina de Itabira


In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from src.ingestion import load_telemetry, cast_telemetry_types, get_telemetry_stats

ROOT = Path("..")
GLOB = str(ROOT / "Base_Dados/datasets/telemetria/*.parquet")
con = duckdb.connect()

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(50)


polars.config.Config

## 1. Visão Geral do Dataset

In [2]:
stats = get_telemetry_stats()
print(stats)


shape: (6, 6)
┌─────┬───────────┬─────────────────┬─────────────┬───────────────┬─────────────────────────┐
│ mes ┆ registros ┆ dont_go_eventos ┆ dont_go_pct ┆ data_inicio   ┆ data_fim                │
│ --- ┆ ---       ┆ ---             ┆ ---         ┆ ---           ┆ ---                     │
│ str ┆ i64       ┆ decimal[38,0]   ┆ f64         ┆ datetime[μs]  ┆ datetime[μs]            │
╞═════╪═══════════╪═════════════════╪═════════════╪═══════════════╪═════════════════════════╡
│ jan ┆ 5400002   ┆ 2581            ┆ 0.0478      ┆ 2025-01-01    ┆ 2025-01-31 23:59:58.737 │
│     ┆           ┆                 ┆             ┆ 00:00:00.017  ┆                         │
│ feb ┆ 5709935   ┆ 4493            ┆ 0.0787      ┆ 2025-02-01    ┆ 2025-02-28 23:59:54.920 │
│     ┆           ┆                 ┆             ┆ 00:00:03.657  ┆                         │
│ mar ┆ 5688538   ┆ 4223            ┆ 0.0742      ┆ 2025-03-01    ┆ 2025-03-31 23:59:58.827 │
│     ┆           ┆                 ┆         

In [3]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=["Jan","Fev","Mar","Abr","Mai","Jun"],
    y=stats["registros"].to_list(),
    name="Total de Eventos", marker_color="#1f77b4", opacity=0.7
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=["Jan","Fev","Mar","Abr","Mai","Jun"],
    y=stats["dont_go_eventos"].cast(pl.Int64).to_list(),
    name="Eventos Don't Go", mode="lines+markers", marker_color="#d62728", line_width=2
), secondary_y=True)

fig.update_layout(
    title="Volume de Eventos de Telemetria por Mês (barra) e Don't Go (linha)",
    height=420
)
fig.update_yaxes(title_text="Total de Eventos", secondary_y=False)
fig.update_yaxes(title_text="Eventos Don't Go", secondary_y=True)
fig.show()


## 2. Distribuição por Criticidade

In [4]:
# Normalizar string de Criticidade (problemas de encoding no dado bruto)
crit = con.execute(f'''
    SELECT
        Id_Criticidade,
        CASE Id_Criticidade
            WHEN 1 THEN 'Crítico'
            WHEN 2 THEN 'Não Crítico'
            WHEN 3 THEN 'Informacional'
            WHEN 4 THEN 'Outro'
        END AS Criticidade,
        COUNT(*) AS total,
        SUM(Is_Dont_Go) AS dont_go
    FROM read_parquet('{GLOB}')
    GROUP BY Id_Criticidade
    ORDER BY Id_Criticidade
''').pl()

crit = crit.with_columns(
    (pl.col("total") / pl.col("total").sum() * 100).round(2).alias("pct_total")
)
print(crit)


shape: (4, 5)
┌────────────────┬───────────────┬──────────┬───────────────┬───────────┐
│ Id_Criticidade ┆ Criticidade   ┆ total    ┆ dont_go       ┆ pct_total │
│ ---            ┆ ---           ┆ ---      ┆ ---           ┆ ---       │
│ i64            ┆ str           ┆ i64      ┆ decimal[38,0] ┆ f64       │
╞════════════════╪═══════════════╪══════════╪═══════════════╪═══════════╡
│ 1              ┆ Crítico       ┆ 83020    ┆ 10286         ┆ 0.22      │
│ 2              ┆ Não Crítico   ┆ 461865   ┆ 9676          ┆ 1.24      │
│ 3              ┆ Informacional ┆ 36616050 ┆ 0             ┆ 98.53     │
│ 4              ┆ Outro         ┆ 3119     ┆ 0             ┆ 0.01      │
└────────────────┴───────────────┴──────────┴───────────────┴───────────┘


In [5]:
fig = px.pie(
    crit.to_pandas(),
    names="Criticidade", values="total",
    title="Distribuição de Eventos por Nível de Criticidade (37M registros)",
    hole=0.45,
    color="Criticidade",
    color_discrete_map={
        "Crítico": "#d62728",
        "Não Crítico": "#ff7f0e",
        "Informacional": "#1f77b4",
        "Outro": "#9467bd"
    }
)
fig.update_traces(textinfo="label+percent")
fig.show()


## 3. Alarmes Don't Go — Fingerprint

**Descoberta crítica:** todos os eventos marcados `Is_Dont_Go=1` correspondem a alarmes específicos com **100% de correlação** com o estado Don't Go. Esses são os alarmes que *compõem* o evento, não que o *precedem*.

O desafio preditivo real: identificar a sequência de alarmes que **precede** esses eventos nas horas anteriores.


In [6]:
dontgo_alarms = con.execute(f'''
    SELECT
        Id_Alarme,
        Alarme,
        CASE Id_Criticidade
            WHEN 1 THEN 'Crítico'
            WHEN 2 THEN 'Não Crítico'
            ELSE 'Outro'
        END AS Criticidade,
        COUNT(*) AS total_eventos,
        COUNT(DISTINCT TAG) AS n_equipamentos
    FROM read_parquet('{GLOB}')
    WHERE Is_Dont_Go = 1
    GROUP BY Id_Alarme, Alarme, Id_Criticidade
    ORDER BY total_eventos DESC
''').pl()

print(f"Tipos únicos de alarme Don't Go: {len(dontgo_alarms)}")
print(dontgo_alarms)


Tipos únicos de alarme Don't Go: 68
shape: (68, 5)
┌────────────┬──────────────────────────────────────┬─────────────┬───────────────┬────────────────┐
│ Id_Alarme  ┆ Alarme                               ┆ Criticidade ┆ total_eventos ┆ n_equipamentos │
│ ---        ┆ ---                                  ┆ ---         ┆ ---           ┆ ---            │
│ i64        ┆ str                                  ┆ str         ┆ i64           ┆ i64            │
╞════════════╪══════════════════════════════════════╪═════════════╪═══════════════╪════════════════╡
│ 84608753   ┆ Engine Coolant Level - Active        ┆ Não Crítico ┆ 7505          ┆ 24             │
│ 84609435   ┆ Right Front Brake Temperature -      ┆ Crítico     ┆ 4119          ┆ 1              │
│            ┆ Active                               ┆             ┆               ┆                │
│ 84613669   ┆ Transmission Oil Level - Active      ┆ Crítico     ┆ 1426          ┆ 26             │
│ 84608753   ┆ Engine Coolant Level - Ac

In [7]:
fig = px.bar(
    dontgo_alarms.to_pandas(),
    x="total_eventos", y="Alarme", orientation="h",
    color="Criticidade",
    title="Alarmes que Compõem Eventos Don't Go (Alarm Fingerprint)",
    labels={"total_eventos": "Total de Ocorrências", "Alarme": ""},
    text="n_equipamentos",
    color_discrete_map={"Crítico": "#d62728", "Não Crítico": "#ff7f0e"},
)
fig.update_traces(texttemplate="%{text} equip.", textposition="outside")
fig.update_layout(height=550, yaxis={"categoryorder": "total ascending"})
fig.show()


## 4. Padrão Temporal dos Eventos Don't Go

In [8]:
# Don't Go por hora do dia
hora_dg = con.execute(f'''
    SELECT
        HOUR(Data_Evento) AS hora,
        COUNT(*) AS total_eventos,
        SUM(Is_Dont_Go) AS dont_go,
        ROUND(100.0 * SUM(Is_Dont_Go) / COUNT(*), 4) AS taxa_dg
    FROM read_parquet('{GLOB}')
    GROUP BY hora
    ORDER BY hora
''').pl()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=hora_dg["hora"].to_list(),
    y=hora_dg["total_eventos"].to_list(),
    name="Total Eventos", marker_color="#1f77b4", opacity=0.5
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=hora_dg["hora"].to_list(),
    y=hora_dg["taxa_dg"].to_list(),
    name="Taxa Don't Go (%)", mode="lines+markers", marker_color="#d62728"
), secondary_y=True)
fig.update_layout(title="Distribuição por Hora do Dia (H4 — Turnos)", height=400)
fig.update_yaxes(title_text="Total de Eventos", secondary_y=False)
fig.update_yaxes(title_text="Taxa Don't Go (%)", secondary_y=True)
fig.show()


In [9]:
# Don't Go por mês (evolução)
mes_dg = con.execute(f'''
    SELECT
        MONTH(Data_Evento) AS mes,
        COUNT(*) AS total,
        SUM(Is_Dont_Go) AS dont_go,
        ROUND(100.0 * SUM(Is_Dont_Go) / COUNT(*), 4) AS taxa_dg
    FROM read_parquet('{GLOB}')
    GROUP BY mes ORDER BY mes
''').pl()

meses = ["Jan","Fev","Mar","Abr","Mai","Jun"]
fig = px.bar(
    mes_dg.to_pandas(), x="mes", y="dont_go",
    title="Eventos Don't Go por Mês — H4: variação temporal",
    labels={"dont_go": "Eventos Don't Go", "mes": "Mês"},
    text="dont_go", color="taxa_dg",
    color_continuous_scale="Reds",
)
fig.update_xaxes(tickvals=list(range(1,7)), ticktext=meses)
fig.update_traces(textposition="outside")
fig.update_layout(height=400)
fig.show()


## 5. Perfil de Don't Go por Equipamento (H3 — Frotas)

In [10]:
equip_dg = con.execute(f'''
    SELECT
        TAG, Tag_Frota,
        COUNT(*) AS total_eventos,
        SUM(Is_Dont_Go) AS dont_go,
        ROUND(100.0 * SUM(Is_Dont_Go) / COUNT(*), 3) AS taxa_dg
    FROM read_parquet('{GLOB}')
    GROUP BY TAG, Tag_Frota
    ORDER BY dont_go DESC
''').pl()

print(equip_dg)


shape: (35, 5)
┌─────────┬───────────────────┬───────────────┬───────────────┬─────────┐
│ TAG     ┆ Tag_Frota         ┆ total_eventos ┆ dont_go       ┆ taxa_dg │
│ ---     ┆ ---               ┆ ---           ┆ ---           ┆ ---     │
│ str     ┆ str               ┆ i64           ┆ decimal[38,0] ┆ f64     │
╞═════════╪═══════════════════╪═══════════════╪═══════════════╪═════════╡
│ CA65926 ┆ 793-D 4S          ┆ 95554         ┆ 4923          ┆ 5.152   │
│ CA65931 ┆ 793-D 5S          ┆ 158610        ┆ 1710          ┆ 1.078   │
│ CA65930 ┆ 793-D 5S          ┆ 214525        ┆ 1622          ┆ 0.756   │
│ CA65792 ┆ 793-D 2S          ┆ 126551        ┆ 1589          ┆ 1.256   │
│ CA65927 ┆ 793-D 5S          ┆ 87988         ┆ 1313          ┆ 1.492   │
│ CA65925 ┆ 793-D 4S          ┆ 174146        ┆ 915           ┆ 0.525   │
│ CA65937 ┆ 793-D 5S          ┆ 94629         ┆ 740           ┆ 0.782   │
│ CA65934 ┆ 793-D 5S          ┆ 100484        ┆ 705           ┆ 0.702   │
│ CA65908 ┆ 793-D 3S   

In [11]:
fig = px.scatter(
    equip_dg.to_pandas(),
    x="total_eventos", y="dont_go",
    color="Tag_Frota", hover_name="TAG",
    size="taxa_dg",
    title="Don't Go vs Volume de Eventos por Equipamento (H3 — Perfil por Frota)",
    labels={"total_eventos": "Total de Eventos", "dont_go": "Eventos Don't Go", "Tag_Frota": "Frota"},
)
fig.update_layout(height=500)
fig.show()


In [12]:
# Taxa média de Don't Go por frota (H3)
frota_dg = con.execute(f'''
    SELECT
        Tag_Frota,
        COUNT(*) AS total,
        SUM(Is_Dont_Go) AS dont_go,
        ROUND(100.0 * SUM(Is_Dont_Go) / COUNT(*), 4) AS taxa_dg,
        COUNT(DISTINCT TAG) AS n_equipamentos
    FROM read_parquet('{GLOB}')
    GROUP BY Tag_Frota
    ORDER BY taxa_dg DESC
''').pl()
print(frota_dg)

fig = px.bar(
    frota_dg.to_pandas(),
    x="Tag_Frota", y="taxa_dg", text="n_equipamentos",
    title="Taxa de Don't Go (%) por Modelo de Frota — H3",
    labels={"taxa_dg": "Taxa Don't Go (%)", "Tag_Frota": "Frota"},
    color="taxa_dg", color_continuous_scale="Reds",
)
fig.update_traces(texttemplate="%{text} equip.", textposition="outside")
fig.update_layout(height=420)
fig.show()


shape: (5, 5)
┌───────────────────┬──────────┬───────────────┬─────────┬────────────────┐
│ Tag_Frota         ┆ total    ┆ dont_go       ┆ taxa_dg ┆ n_equipamentos │
│ ---               ┆ ---      ┆ ---           ┆ ---     ┆ ---            │
│ str               ┆ i64      ┆ decimal[38,0] ┆ f64     ┆ i64            │
╞═══════════════════╪══════════╪═══════════════╪═════════╪════════════════╡
│ 793-D 4S          ┆ 854298   ┆ 7405          ┆ 0.8668  ┆ 9              │
│ 793-D 3S          ┆ 165130   ┆ 1350          ┆ 0.8175  ┆ 3              │
│ 793-D 5S          ┆ 1881355  ┆ 9341          ┆ 0.4965  ┆ 13             │
│ 793-D 2S          ┆ 380980   ┆ 1699          ┆ 0.446   ┆ 5              │
│ LeTourneau L 1850 ┆ 33882291 ┆ 167           ┆ 0.0005  ┆ 5              │
└───────────────────┴──────────┴───────────────┴─────────┴────────────────┘


## 6. Frequência de Alarmes Críticos (H1 e H2 — Precursores Don't Go)

In [13]:
# Top alarmes críticos que aparecem ANTES de eventos Don't Go
# Para validar H1/H2, vamos analisar os alarmes críticos que precedem os Don't Go
# carregando jan em memória como amostra
lf = cast_telemetry_types(load_telemetry(["jan", "feb"]))
df = lf.select([
    "TAG", "Data_Evento", "Id_Alarme", "Alarme",
    "Id_Criticidade", "Criticidade", "Is_Dont_Go"
]).collect()

print(f"Registros carregados: {len(df):,}")
print(f"Don't Go nessa amostra: {df['Is_Dont_Go'].sum():,}")


Registros carregados: 11,109,937
Don't Go nessa amostra: 7,074


In [14]:
# Para cada evento Don't Go, encontrar os alarmes nas 4h anteriores no mesmo equipamento
# Usamos DuckDB com window function para eficiência
import tempfile, os

# Salvar amostra temporária
tmp_path = "/tmp/telemetria_jan_feb.parquet"
df.write_parquet(tmp_path)

precursores = con.execute(f'''
WITH dont_go_events AS (
    SELECT TAG, Data_Evento AS dg_time
    FROM read_parquet('{tmp_path}')
    WHERE Is_Dont_Go = 1
),
preceding AS (
    SELECT
        d.TAG,
        d.dg_time,
        t.Id_Alarme,
        t.Alarme,
        t.Id_Criticidade,
        (EPOCH(d.dg_time) - EPOCH(t.Data_Evento)) / 60.0 AS minutos_antes
    FROM dont_go_events d
    JOIN read_parquet('{tmp_path}') t
        ON d.TAG = t.TAG
        AND t.Data_Evento < d.dg_time
        AND t.Data_Evento >= d.dg_time - INTERVAL 4 HOUR
        AND t.Is_Dont_Go = 0
)
SELECT
    Id_Alarme, Alarme, Id_Criticidade,
    COUNT(*) AS ocorrencias,
    COUNT(DISTINCT TAG) AS n_equipamentos,
    ROUND(AVG(minutos_antes), 1) AS media_min_antes
FROM preceding
GROUP BY Id_Alarme, Alarme, Id_Criticidade
ORDER BY ocorrencias DESC
LIMIT 20
''').pl()

print("Top alarmes nas 4h ANTES dos Don't Go:")
print(precursores)


Top alarmes nas 4h ANTES dos Don't Go:
shape: (20, 6)
┌────────────┬───────────────────┬────────────────┬─────────────┬────────────────┬─────────────────┐
│ Id_Alarme  ┆ Alarme            ┆ Id_Criticidade ┆ ocorrencias ┆ n_equipamentos ┆ media_min_antes │
│ ---        ┆ ---               ┆ ---            ┆ ---         ┆ ---            ┆ ---             │
│ i64        ┆ str               ┆ i8             ┆ i64         ┆ i64            ┆ f64             │
╞════════════╪═══════════════════╪════════════════╪═════════════╪════════════════╪═════════════════╡
│ 84608752   ┆ Engine Coolant    ┆ 3              ┆ 748937      ┆ 24             ┆ 90.2            │
│            ┆ Level - Inactive  ┆                ┆             ┆                ┆                 │
│ 84608753   ┆ Engine Coolant    ┆ 2              ┆ 499698      ┆ 20             ┆ 90.5            │
│            ┆ Level - Active    ┆                ┆             ┆                ┆                 │
│ 1241579532 ┆ Engine \/         ┆ 3 

In [15]:
fig = px.bar(
    precursores.head(15).to_pandas(),
    x="ocorrencias", y="Alarme", orientation="h",
    color="Id_Criticidade",
    title="Top 15 Alarmes Precursores nas 4h Antes de Don't Go (jan+fev) — H1/H2",
    labels={"ocorrencias": "Ocorrências", "Alarme": ""},
    color_continuous_scale="Reds",
    text="n_equipamentos",
)
fig.update_traces(texttemplate="%{text} equip.", textposition="outside")
fig.update_layout(height=520, yaxis={"categoryorder": "total ascending"})
fig.show()


## 7. Validação das Hipóteses de Negócio

In [16]:
# H5: manutenção recente influencia Don't Go?
# Cruzar com apontamentos no transformation.py — registrado como pendente

# Resumo das hipóteses validadas neste notebook
hipoteses = {
    "H1": ("Sequência característica de alarmes precede Don't Go em até 4h",
           "PARCIALMENTE VALIDADA", "Alarmes informacionais de alta frequência precedem o evento"),
    "H2": ("Frequência de alarmes críticos aumenta antes do Don't Go",
           "A VALIDAR no feature engineering", "Necessita janelas temporais sliding window"),
    "H3": ("Frotas 793-D têm perfis de falha distintos",
           "VALIDADA", "Taxa DG varia por frota: 793-D 2S lidera, mas volume difere"),
    "H4": ("Diferença de alarmes entre turnos dia vs noite",
           "NÃO CONFIRMADA", "Distribuição horária de Don't Go é uniforme"),
    "H5": ("Estado de manutenção influencia Don't Go",
           "PENDENTE", "Requer join com apontamentos no próximo módulo"),
    "H6": ("Localidades com maior concentração de alarmes",
           "N/A", "Apenas Itabira nos dados — variável constante"),
    "H7": ("Correlação operador-equipamento com perfil de alarmes",
           "PENDENTE", "Colunas de operador ausentes no parquet de apontamentos"),
}

for h, (descricao, status, nota) in hipoteses.items():
    print(f"[{h}] {status}")
    print(f"     {descricao}")
    print(f"     → {nota}")
    print()


[H1] PARCIALMENTE VALIDADA
     Sequência característica de alarmes precede Don't Go em até 4h
     → Alarmes informacionais de alta frequência precedem o evento

[H2] A VALIDAR no feature engineering
     Frequência de alarmes críticos aumenta antes do Don't Go
     → Necessita janelas temporais sliding window

[H3] VALIDADA
     Frotas 793-D têm perfis de falha distintos
     → Taxa DG varia por frota: 793-D 2S lidera, mas volume difere

[H4] NÃO CONFIRMADA
     Diferença de alarmes entre turnos dia vs noite
     → Distribuição horária de Don't Go é uniforme

[H5] PENDENTE
     Estado de manutenção influencia Don't Go
     → Requer join com apontamentos no próximo módulo

[H6] N/A
     Localidades com maior concentração de alarmes
     → Apenas Itabira nos dados — variável constante

[H7] PENDENTE
     Correlação operador-equipamento com perfil de alarmes
     → Colunas de operador ausentes no parquet de apontamentos



## 8. Insights e Conclusões

### Achados Críticos

| # | Insight | Impacto no Modelo |
|---|---------|-------------------|
| I1 | **Is_Dont_Go = 1 é determinístico**: todos os alarmes com essa flag têm 100% de taxa DG — são os alarmes *que definem* o evento, não precursores | O target real é prever esses alarmes **antes** de ocorrerem |
| I2 | **Desbalanceamento severo**: 0,054% de eventos são Don't Go (1:1.860) | Obrigatório usar `scale_pos_weight` e otimizar F1/PR-AUC |
| I3 | **Apenas Itabira** nos dados — localidade é variável constante | Remover da feature matrix |
| I4 | **CA65926 (793-D 4S)** tem 4.923 DG em 95K eventos = taxa de 5,15% — outlier significativo | Tratar como equipamento de alto risco, analisar separadamente |
| I5 | **H4 refutada**: Don't Go é uniformemente distribuído ao longo das 24h | Hora do dia não é feature discriminativa para DG |
| I6 | **Encoding issues** em `Criticidade` (string com caracteres corrompidos) | Usar `Id_Criticidade` como referência, não o texto |
| I7 | **Alarmes informacionais (Id=3)** são 98,5% dos dados — base do fingerprint | Features de frequência devem ser por Id_Alarme, não só por criticidade |

### Próximos Passos
- `src/transformation.py`: join telemetria ↔ apontamentos + flag `is_pre_dont_go` (janelas 1h e 4h antes)
- `04_feature_engineering.ipynb`: sliding window features por equipamento e janela temporal
- Validar H2 e H5 com pipeline de features
